# Like-Day Model — Scenario Comparison Dashboard

Pull completed W&B runs and visualize scenario inputs/outputs with plotly.

**Modes:**
- **Grid search**: Compare hyperparameter configs for a single date
- **Backtest**: Compare one config across multiple dates
- **Backtest grid**: Cross-date × cross-config comparison

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import wandb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

WANDB_PROJECT = "pjm-like-day-forecast"
api = wandb.Api()

## 1. Pull Runs from W&B

Filter by sweep ID, tag, or date range. Adjust the filter below.

In [ ]:
# ── Configure which runs to pull ──
# Option A: Pull all runs from a sweep
# SWEEP_ID = "your-entity/pjm-like-day-forecast/SWEEP_ID"
# runs = api.sweep(SWEEP_ID).runs

# Option B: Pull by tag
# runs = api.runs(WANDB_PROJECT, filters={"tags": "backtest"})

# Option C: Pull all recent runs
runs = api.runs(WANDB_PROJECT, order="-created_at", per_page=200)

# Build comparison DataFrame
rows = []
for run in runs:
    if run.state != "finished":
        continue
    row = {
        "run_id": run.id,
        "run_name": run.name,
        "group": run.group,
    }
    row.update(run.config)
    row.update(run.summary._json_dict)
    rows.append(row)

df_runs = pd.DataFrame(rows)
print(f"Loaded {len(df_runs)} finished runs")
df_runs.head()

## 2. Parallel Coordinates — Hyperparameter Exploration

See which parameter combinations yield the lowest MAE.

In [ ]:
# Encode weight_method as numeric for parallel coordinates
method_map = {"inverse_distance": 0, "softmax": 1, "rank": 2, "uniform": 3}
df_pc = df_runs.copy()
df_pc["weight_method_num"] = df_pc["weight_method"].map(method_map)

dims = [
    dict(label="n_analogs", values=df_pc["n_analogs"]),
    dict(label="weight_method", values=df_pc["weight_method_num"],
         tickvals=list(method_map.values()), ticktext=list(method_map.keys())),
    dict(label="season_window", values=df_pc["season_window_days"]),
    dict(label="same_dow", values=df_pc["same_dow_group"].astype(int)),
    dict(label="MAE", values=df_pc["mae"]),
    dict(label="RMSE", values=df_pc["rmse"]),
]

fig = go.Figure(data=go.Parcoords(
    line=dict(color=df_pc["mae"], colorscale="Viridis_r", showscale=True,
              cmin=df_pc["mae"].quantile(0.05), cmax=df_pc["mae"].quantile(0.95)),
    dimensions=dims,
))
fig.update_layout(title="Hyperparameter Parallel Coordinates (colored by MAE)", height=500)
fig.show()

## 3. Heatmap — n_analogs vs weight_method

In [ ]:
# Pivot MAE by n_analogs and weight_method (average across dates if backtest)
if "forecast_date" in df_runs.columns and df_runs["forecast_date"].nunique() > 1:
    pivot = df_runs.groupby(["n_analogs", "weight_method"])["mae"].mean().reset_index()
    title_suffix = " (avg across dates)"
else:
    pivot = df_runs[["n_analogs", "weight_method", "mae"]].copy()
    title_suffix = ""

heatmap_data = pivot.pivot(index="weight_method", columns="n_analogs", values="mae")

fig = px.imshow(
    heatmap_data,
    text_auto=".2f",
    color_continuous_scale="RdYlGn_r",
    labels=dict(x="n_analogs", y="weight_method", color="MAE"),
    title=f"MAE by n_analogs x weight_method{title_suffix}",
)
fig.update_layout(height=400)
fig.show()

## 4. Backtest Time Series — MAE Over Forecast Dates

In [ ]:
if "forecast_date" in df_runs.columns and df_runs["forecast_date"].nunique() > 1:
    df_bt = df_runs.sort_values("forecast_date")

    # Build a config label for each unique config
    df_bt["config_label"] = (
        "n" + df_bt["n_analogs"].astype(str) + "_" +
        df_bt["weight_method"] + "_sw" +
        df_bt["season_window_days"].astype(str)
    )

    fig = px.line(
        df_bt, x="forecast_date", y="mae",
        color="config_label",
        markers=True,
        title="Backtest: MAE by Forecast Date",
        labels={"mae": "MAE ($/MWh)", "forecast_date": "Forecast Date"},
    )
    fig.update_layout(height=450)
    fig.show()

    # RMSE too
    if "rmse" in df_bt.columns:
        fig2 = px.line(
            df_bt, x="forecast_date", y="rmse",
            color="config_label",
            markers=True,
            title="Backtest: RMSE by Forecast Date",
            labels={"rmse": "RMSE ($/MWh)"},
        )
        fig2.update_layout(height=450)
        fig2.show()
else:
    print("Single forecast date — skip backtest time series.")

## 5. Best vs Worst Run — Hourly Profile Comparison

In [ ]:
# Find best and worst runs by MAE
df_with_mae = df_runs.dropna(subset=["mae"])
if len(df_with_mae) >= 2:
    best_run_id = df_with_mae.loc[df_with_mae["mae"].idxmin(), "run_id"]
    worst_run_id = df_with_mae.loc[df_with_mae["mae"].idxmax(), "run_id"]

    def pull_hourly_table(run_id: str) -> pd.DataFrame | None:
        """Pull the hourly_forecast wandb.Table for a given run."""
        wb_run = api.run(f"{WANDB_PROJECT}/{run_id}")
        for artifact in wb_run.logged_artifacts():
            for f in artifact.files():
                if "hourly_forecast" in f.name:
                    table = artifact.get(f.name)
                    if table is not None:
                        return pd.DataFrame(data=table.data, columns=table.columns)
        # Fall back to history
        history = wb_run.history()
        if "forecast_lmp" in history.columns:
            return history[["forecast_lmp", "actual_lmp"]].dropna()
        return None

    best_info = df_with_mae.loc[df_with_mae["mae"].idxmin()]
    worst_info = df_with_mae.loc[df_with_mae["mae"].idxmax()]

    print(f"Best:  {best_info['run_name']}  MAE=${best_info['mae']:.2f}")
    print(f"Worst: {worst_info['run_name']} MAE=${worst_info['mae']:.2f}")
else:
    print("Need at least 2 runs with MAE to compare.")

## 6. Metric Radar Chart — Multi-metric Comparison

In [ ]:
# Compare top 5 configs across multiple metrics
metric_cols = [c for c in ["mae", "rmse", "mape", "crps", "mean_pinball"]
               if c in df_runs.columns]

if len(metric_cols) >= 3 and len(df_with_mae) >= 2:
    top5 = df_with_mae.nsmallest(5, "mae")

    # Normalize each metric to [0, 1] range for radar
    df_norm = top5[metric_cols].copy()
    for col in metric_cols:
        col_min, col_max = df_runs[col].min(), df_runs[col].max()
        if col_max > col_min:
            df_norm[col] = (df_norm[col] - col_min) / (col_max - col_min)
        else:
            df_norm[col] = 0.5

    fig = go.Figure()
    for i, (idx, row) in enumerate(df_norm.iterrows()):
        run_name = top5.loc[idx, "run_name"]
        values = row[metric_cols].tolist()
        values.append(values[0])  # close the polygon
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=metric_cols + [metric_cols[0]],
            name=run_name,
            fill="toself",
            opacity=0.6,
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Top 5 Configs — Normalized Metric Radar (lower = better)",
        height=500,
    )
    fig.show()
else:
    print(f"Need >= 3 metrics and >= 2 runs. Have {len(metric_cols)} metrics, {len(df_with_mae)} runs.")

## 7. Summary Table — Ranked by MAE

In [ ]:
summary_cols = ["run_name", "n_analogs", "weight_method", "season_window_days",
                "same_dow_group"]
summary_cols += [c for c in ["forecast_date", "mae", "rmse", "mape", "crps",
                              "coverage_90pct", "mean_pinball"]
                 if c in df_runs.columns]

available_cols = [c for c in summary_cols if c in df_runs.columns]
df_summary = df_runs[available_cols].sort_values("mae").reset_index(drop=True)
df_summary.index += 1
df_summary.index.name = "rank"
df_summary.head(20)

## 8. Aggregate Backtest Statistics

In [ ]:
if "forecast_date" in df_runs.columns and df_runs["forecast_date"].nunique() > 1:
    df_bt = df_runs.copy()
    df_bt["config_label"] = (
        "n" + df_bt["n_analogs"].astype(str) + "_" +
        df_bt["weight_method"] + "_sw" +
        df_bt["season_window_days"].astype(str)
    )

    agg_metrics = [c for c in ["mae", "rmse", "mape", "crps"] if c in df_bt.columns]

    agg = df_bt.groupby("config_label")[agg_metrics].agg(["mean", "std", "min", "max"])
    agg.columns = [f"{m}_{stat}" for m, stat in agg.columns]
    agg = agg.sort_values("mae_mean")

    print("\nAggregate Backtest Statistics (sorted by mean MAE):")
    display(agg.style.format("{:.3f}"))

    # Box plot of MAE by config
    fig = px.box(
        df_bt, x="config_label", y="mae",
        title="MAE Distribution by Config (across backtest dates)",
        labels={"mae": "MAE ($/MWh)", "config_label": "Configuration"},
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print("Single forecast date — aggregate stats not applicable.")